# Meeting Minutes Generator

### ==========================================================
### MP3 -> Transcript (Whisper) -> High-quality Minutes (Ollama)
### ==========================================================

#### ---- Import Section ----

In [1]:
import torch                         # torch = PyTorch library (used here for GPU/CPU detection + dtype like float16/float32)
import librosa                       # librosa = audio loading/processing library (loads MP3 into arrays)
from transformers import pipeline    # pipeline = easy Hugging Face wrapper (we use it for Whisper ASR transcription)
from openai import OpenAI            # OpenAI client (we use it with Ollama's OpenAI-compatible API)
from IPython.display import display, Markdown  # display/Markdown = show Markdown output nicely inside Jupyter Notebook

#### -----------------------------
#### 0) SETTINGS (change only these)
#### -----------------------------


In [2]:
# audio_file_path:
# - "r" before the string means "raw string"
# - raw string avoids issues with backslashes in Windows paths (\)
audio_file_path = r"D:\work\llm_engineering\Data\denver_extract.mp3"  # path to your input MP3 file

# whisper_model_name:
# - Whisper model for speech-to-text (ASR)
# - base.en is a good balance; tiny.en is faster but less accurate
whisper_model_name = "openai/whisper-base.en"

# minutes_model_name:
# - the LLM you will call via Ollama
minutes_model_name = "gpt-oss:120b-cloud"

# max_transcript_characters:
# - limits transcript length to keep LLM generation stable and faster
#max_transcript_characters = 14000

# chunk_length_seconds:
# - Whisper will process audio in chunks of this many seconds
# - helps with long audio and memory usage
chunk_length_seconds = 45

# Map-Reduce Specific Settings
# We break the transcript into chunks of 6,000 characters to ensure the LLM stays focused.
llm_chunk_size = 6000 

#### ==========================================================
#### 1) GPU or CPU?
#### ==========================================================

In [3]:
# is_gpu_available:
# - torch.cuda.is_available() returns True if CUDA GPU is available, else False
is_gpu_available = torch.cuda.is_available()

# device_number:
# - transformers pipeline uses:
#   device=0  -> first GPU (cuda:0)
#   device=-1 -> CPU
device_number = 0 if is_gpu_available else -1  # "if GPU then 0 else -1"

# Print whether CUDA is available
print("CUDA available:", is_gpu_available)

# If GPU exists, print GPU name (example: "NVIDIA GeForce GTX 1650")
if is_gpu_available:
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce GTX 1650


#### ==========================================================
#### 2) Load audio (Whisper expects 16kHz mono)
#### ==========================================================

In [4]:
print("\n[1/3] Loading audio...")  # \n makes a blank line before the text

# librosa.load(...) returns TWO things:
# 1) audio_data = waveform array (numbers representing sound amplitude)
# 2) sample_rate = samples per second
#
# We force:
# - sr=16000   -> resample audio to 16kHz (Whisper expects 16kHz)
# - mono=True  -> convert to mono (single channel)
audio_data, sample_rate = librosa.load(audio_file_path, sr=16000, mono=True)

# len(audio_data) = number of samples
# sample_rate = samples per second
# len(audio_data)/sample_rate = total seconds
print("Audio length (sec):", round(len(audio_data) / sample_rate, 1))  # round(..., 1) = 1 decimal place


[1/3] Loading audio...
Audio length (sec): 900.0


#### ==========================================================
#### 3) Transcribe with Whisper (with timestamps)
#### ==========================================================

In [5]:
print("[2/3] Transcribing with Whisper...")

# speech_to_text_model = pipeline(...):
# - task: "automatic-speech-recognition" = speech to text
# - model: whisper_model_name
# - device: GPU/CPU selection
# - torch_dtype:
#     float16 on GPU (faster + less memory)
#     float32 on CPU (safe/default)
# - return_timestamps=True:
#     asks Whisper to return timestamps for parts of transcript
# - chunk_length_s:
#     audio is processed in segments
# - stride_length_s=(2,2):
#     overlap between chunks (left and right) to reduce cut-off words
speech_to_text_model = pipeline(
    "automatic-speech-recognition",              # task name
    model=whisper_model_name,                    # whisper model ID
    device=device_number,                        # 0 for GPU, -1 for CPU
    torch_dtype=torch.float16 if is_gpu_available else torch.float32,  # pick dtype based on GPU availability
    return_timestamps=True,                      # get timestamps in output
    chunk_length_s=chunk_length_seconds,         # chunk size in seconds
    stride_length_s=(2, 2),                      # overlap in seconds (left, right)
)

# transcription_result:
# - calling the pipeline like a function runs ASR
# - input is audio_data (waveform array)
transcription_result = speech_to_text_model(audio_data)

# convert_seconds_to_mmss(seconds_value):
# - converts time in seconds to "MM:SS"
# - handles tuple/list timestamps too (sometimes timestamp is (start,end))
def convert_seconds_to_mmss(seconds_value):
    """Convert seconds -> MM:SS (handles tuple timestamps too)."""

    # If timestamp is like (start, end), take the start time
    if isinstance(seconds_value, (tuple, list)):  # isinstance checks the type
        seconds_value = seconds_value[0]

    # If timestamp is missing (None), return placeholder
    if seconds_value is None:
        return "??:??"

    # Convert to integer seconds (remove decimals)
    seconds_value = int(seconds_value)

    # divmod(a, b) returns (a//b, a%b)
    # Here: minutes = seconds_value//60, seconds = seconds_value%60
    minutes, seconds = divmod(seconds_value, 60)

    # f-string formatting:
    # {minutes:02d} means "2 digits, pad with 0"
    return f"{minutes:02d}:{seconds:02d}"




[2/3] Transcribing with Whisper...


`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


In [6]:
# Build timestamped transcript (one line per chunk)
# Whisper pipeline can return output as:
# - transcription_result["chunks"] : list of chunk objects with "text" and "timestamp"
# OR
# - transcription_result["text"]   : plain text string (no chunks)
# Build the Full Transcript String
full_transcript = ""
if "chunks" in transcription_result:
    lines = [f"[{convert_seconds_to_mmss(c.get('timestamp'))}] {c.get('text').strip()}" for c in transcription_result["chunks"]]
    full_transcript = "\n".join(lines)
else:
    full_transcript = transcription_result.get("text", "")

print(f"Transcription complete. Total characters: {len(full_transcript)}")

# --- [3/3] GENERATING MINUTES (MAP-REDUCE) ---
print("[3/3] Generating minutes via Map-Reduce logic...\n")

ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# 1. THE "MAP" PHASE: Summarize chunks
# Split the transcript into manageable pieces
transcript_chunks = [full_transcript[i:i + llm_chunk_size] for i in range(0, len(full_transcript), llm_chunk_size)]
intermediate_summaries = []

print(f"Dividing transcript into {len(transcript_chunks)} segments for intermediate processing...")

for idx, chunk in enumerate(transcript_chunks):
    print(f"  > Summarizing segment {idx+1}/{len(transcript_chunks)}...")
    
    map_response = ollama_client.chat.completions.create(
        model=minutes_model_name,
        messages=[
            {"role": "system", "content": "Summarize the following meeting segment concisely. Retain all key decisions, names, and timestamps."},
            {"role": "user", "content": chunk}
        ],
        temperature=0
    )
    intermediate_summaries.append(map_response.choices[0].message.content)

# 2. THE "REDUCE" PHASE: Final Polish
print("\nFinalizing meeting minutes from summaries...")

combined_summaries = "\n\n".join(intermediate_summaries)

# Show how many characters we will send to LLM
print("Transcription done. Characters used:", len(full_transcript))

Transcription complete. Total characters: 9386
[3/3] Generating minutes via Map-Reduce logic...

Dividing transcript into 2 segments for intermediate processing...
  > Summarizing segment 1/2...
  > Summarizing segment 2/2...

Finalizing meeting minutes from summaries...
Transcription done. Characters used: 9386


#### ==========================================================
#### 4) Generate Meeting Minutes via Ollama (PRO Prompt, no prints)
#### ==========================================================

In [7]:

print("[3/3] Generating meeting minutes via Ollama...\n")

# ollama_client:
# - OpenAI client object
# - base_url points to local Ollama OpenAI-compatible server
# - api_key is required by client but Ollama ignores it (placeholder)
#ollama_client = OpenAI(
#    base_url="http://localhost:11434/v1",
#    api_key="ollama"  # placeholder for Ollama
#)

# system_prompt:
# - system message tells the model "who it is" and strict rules
# - this heavily controls formatting + quality
system_prompt = """
You are an expert corporate secretary and council/board meeting scribe.

Your job:
Turn the transcript into clear, accurate, professional meeting minutes.

Output rules (STRICT):
- Output ONLY Markdown minutes (no preface, no explanation, no transcript)
- Do NOT copy transcript lines verbatim; rewrite and summarize (3-4 sentences)
- Keep facts faithful; do NOT invent details
- If something is unclear/missing, write 'Not specified'
- If no action items are explicitly assigned, write 'None mentioned'
- Use blank lines between sections
- Prefer concise bullets; use short paragraphs only where requested

Quality requirements:
- Write a descriptive Meeting Summary (6–8 sentences) explaining context, purpose, and outcomes
- Agenda should be inferred as best as possible from transcript
- Key Discussion Points should be grouped by topic, each topics should have 2–4 bullet points explanation.
- Each bullet should include a timestamp like [MM:SS] when relevant
- Decisions must be explicit; if implied but not confirmed, put under Key Discussion Points instead

Use EXACT headings and order:

# Meeting Summary
# Attendees
# Agenda
# Key Discussion Points
# Decisions Made
# Action Items
# Votes / Motions
# Risks / Blockers
# Next Steps

Attendees:
- Only include names/roles that are clearly mentioned; otherwise write 'Not specified'

Votes / Motions:
- If none clearly stated, write 'None mentioned'
""".strip()  # .strip() removes extra whitespace/newlines at start/end

# user_prompt:
# - includes the transcript and tells the model what to do
# - f-string inserts full_transcript into the message
user_prompt = f"""
Transcript (with timestamps):
{full_transcript}

Now generate the meeting minutes.
""".strip()

# response_stream:
# - chat.completions.create(...) sends messages to the model
# - model=minutes_model_name chooses which Ollama model to use
# - temperature=0 makes output more deterministic (less random)
# - stream=True returns tokens gradually (streaming)
final_response = ollama_client.chat.completions.create(
    model=minutes_model_name,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0
)

generated_minutes = final_response.choices[0].message.content

[3/3] Generating meeting minutes via Ollama...



#### ==========================================================
#### 5) Display Nicely + Save to .md file
#### ==========================================================

In [8]:
# Display Markdown nicely in Jupyter Notebook
display(Markdown(generated_minutes))

# Save to a Markdown file in the current working directory
# with open(...) as file:
# - opens file for writing ("w")
# - encoding="utf-8" supports all characters safely

'''

with open("meeting_minutes.md", "w", encoding="utf-8") as file:
    file.write(generated_minutes)  # write the generated text into the file

print("Saved as: meeting_minutes.md")  # confirm saved file name

'''


# Meeting Summary
The City and County of Denver council convened to review prior meeting minutes, share community announcements, and formally recognize Indigenous Peoples Day. The session opened with an overview of the “Confluence Week” branding, emphasizing water symbolism and Indigenous heritage. Council members approved the October 2 minutes and announced the inaugural Broadway Halloween Parade slated for October 8. A proclamation honoring Indigenous peoples and their contributions was read and adopted, underscoring cultural preservation and environmental stewardship. Several members expressed pride in the new logo and highlighted the importance of cultural inclusivity. No objections or procedural motions were recorded. The meeting concluded with plans to distribute the proclamation and continue engagement with Indigenous communities.

# Attendees
- Councilman Lopez  
- Councilman Clark  
- Councilman Martega (name inferred from transcript)  
- Madam Secretary (Raulkaw)  
- Mr. President (Chair)  
- Not specified (other council members)

# Agenda
- Approval of October 2 minutes  
- Council announcements (including Halloween Parade)  
- Presentation of Indigenous Peoples Day proclamation  
- Discussion of Confluence Week logo and cultural themes  

# Key Discussion Points
- **Minutes Approval** – Members were asked for corrections; none were offered, and the October 2 minutes were approved. [00:45]  
- **Halloween Parade Announcement** – Councilman Clark invited attendees to the first Broadway Halloween Parade on October 8, detailing activities such as candy, “tiki zombies,” and a “fun and funky” atmosphere. [02:53‑02:58]  
- **Proclamation Reading** – Councilman Lopez read Proclamation No. 1127, recognizing the 48 tribal homelands in Denver and declaring Indigenous Peoples Day, with directives for sealing and distribution to relevant agencies. [06:17‑06:33]  
- **Cultural Significance of Logo** – Members praised the new logo’s water symbolism, linked it to land and cultural preservation, and cited a Cesar Chavez quote on cultural pride without contempt. [08:57‑09:21]  
- **Community and Environmental Concerns** – Comments highlighted the need to protect sacred sites and ensure Indigenous cultural continuity amid broader societal challenges. [14:26‑14:44]  

# Decisions Made
- October 2 minutes approved.  
- Council announcements, including the Halloween Parade, accepted.  
- Proclamation for Indigenous Peoples Day adopted and ordered to be sealed and distributed.  

# Action Items
- Councilman Clark to attend and represent the council at the Broadway Halloween Parade on October 8. [02:58]  
- Secretary to affix the City seal to the proclamation and forward copies to the Denver American Indian Commission, Denver School District 1, and the Colorado Commission on Indian Affairs. [06:33]  
- Council members to continue supporting Indigenous cultural events and environmental protection initiatives.  

# Votes / Motions
None mentioned  

# Risks / Blockers
- Potential risk to sacred Indigenous sites and cultural resources noted by participants.  

# Next Steps
- Execute the Halloween Parade on October 8, ensuring community participation.  
- Complete sealing and distribution of the Indigenous Peoples Day proclamation.  
- Plan further activities for Confluence Week to promote Indigenous heritage and environmental awareness.  

'\n\nwith open("meeting_minutes.md", "w", encoding="utf-8") as file:\n    file.write(generated_minutes)  # write the generated text into the file\n\nprint("Saved as: meeting_minutes.md")  # confirm saved file name\n\n'